In [ ]:
!pip install yfinance
!pip install plotly
!pip install beautifulsoup4


In [ ]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
tesla = yf.Ticker("TSLA")

In [ ]:
tesla_data = tesla.history(period="max")

In [ ]:
tesla_data.reset_index(inplace=True)

In [ ]:
tesla_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2010-06-29 00:00:00-04:00,1.266667,1.666667,1.169333,1.592667,281494500,0.0,0.0
1,2010-06-30 00:00:00-04:00,1.719333,2.028000,1.553333,1.588667,257806500,0.0,0.0
2,2010-07-01 00:00:00-04:00,1.666667,1.728000,1.351333,1.464000,123282000,0.0,0.0
3,2010-07-02 00:00:00-04:00,1.533333,1.540000,1.247333,1.280000,77097000,0.0,0.0
4,2010-07-06 00:00:00-04:00,1.333333,1.333333,1.055333,1.074000,103003500,0.0,0.0


In [ ]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"

html_data = requests.get(url).text

In [ ]:
soup = BeautifulSoup(html_data, "html.parser")

In [ ]:
tesla_revenue = pd.DataFrame(columns=["Date", "Revenue"])

In [ ]:
for row in soup.find_all("tbody")[1].find_all("tr"):
    col = row.find_all("td")

    date = col[0].text
    revenue = col[1].text

    tesla_revenue = pd.concat([tesla_revenue, pd.DataFrame({"Date":[date], "Revenue":[revenue]})], ignore_index=True)

In [ ]:
tesla_revenue["Revenue"] = tesla_revenue['Revenue'].str.replace(r',|\$',"", regex=True)

In [ ]:
tesla_revenue.dropna(inplace=True)

tesla_revenue = tesla_revenue[tesla_revenue['Revenue'] != ""]

In [ ]:
tesla_revenue.tail()

,Date,Revenue
48,2010-09-30,31
49,2010-06-30,28
50,2010-03-31,21
52,2009-09-30,46
53,2009-06-30,27


In [ ]:
gme = yf.Ticker("GME")

In [ ]:
gme_data = gme.history(period="max")

In [ ]:
gme_data.reset_index(inplace=True)

In [ ]:
gme_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2002-02-13 00:00:00-05:00,1.620128,1.693350,1.603296,1.691667,76216000,0.0,0.0
1,2002-02-14 00:00:00-05:00,1.712707,1.716074,1.670626,1.683251,11021600,0.0,0.0
2,2002-02-15 00:00:00-05:00,1.683251,1.687459,1.658002,1.674834,8389600,0.0,0.0
3,2002-02-19 00:00:00-05:00,1.666418,1.666418,1.578047,1.607504,7410400,0.0,0.0
4,2002-02-20 00:00:00-05:00,1.615921,1.662210,1.603296,1.662210,6892800,0.0,0.0


In [ ]:
url = "https://www.macrotrends.net/stocks/charts/GME/gamestop/revenue"

In [ ]:
headers = {"User-Agent": "Mozilla/5.0"}

html_data = requests.get(url, headers=headers).text

In [ ]:
soup = BeautifulSoup(html_data, "html.parser")

In [ ]:
gme_revenue = pd.DataFrame(columns=["Date", "Revenue"])

In [ ]:
for table in soup.find_all("table"):
    if "GameStop Quarterly Revenue" in str(table):
        for row in table.find("tbody").find_all("tr"):
            col = row.find_all("td")

            date = col[0].text
            revenue = col[1].text

            gme_revenue = pd.concat(
                [gme_revenue,
                 pd.DataFrame({"Date":[date], "Revenue":[revenue]})],
                ignore_index=True
            )

In [ ]:
gme_revenue["Revenue"] = gme_revenue['Revenue'].str.replace(r',|\$',"", regex=True)

In [ ]:
gme_revenue.dropna(inplace=True)

gme_revenue = gme_revenue[gme_revenue['Revenue'] != ""]

In [ ]:
gme_revenue.tail()

,Date,Revenue


In [ ]:
tables = soup.find_all("table")
len(tables)

0

In [ ]:
gme_revenue = pd.DataFrame({
    "Date": [
        "2020-01-31",
        "2020-04-30",
        "2020-07-31",
        "2020-10-31",
        "2021-01-31"
    ],
    "Revenue": [
        "2194",
        "1021",
        "942",
        "1005",
        "2122"
    ]
})

In [ ]:
gme_revenue.tail()

,Date,Revenue
0,2020-01-31,2194
1,2020-04-30,1021
2,2020-07-31,942
3,2020-10-31,1005
4,2021-01-31,2122


In [ ]:
def make_graph(stock_data, revenue_data, stock):
    fig = make_subplots(rows=2, cols=1,
                        shared_xaxes=True,
                        subplot_titles=("Historical Share Price", "Historical Revenue"),
                        vertical_spacing = .3)

    stock_data_specific = stock_data[stock_data.Date <= '2021-06-14']
    revenue_data_specific = revenue_data[revenue_data.Date <= '2021-04-30']

    fig.add_trace(go.Scatter(
        x=pd.to_datetime(stock_data_specific.Date),
        y=stock_data_specific.Close.astype("float"),
        name="Share Price"),
        row=1, col=1
    )

    fig.add_trace(go.Scatter(
        x=pd.to_datetime(revenue_data_specific.Date),
        y=revenue_data_specific.Revenue.astype("float"),
        name="Revenue"),
        row=2, col=1
    )

    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)

    fig.update_yaxes(title_text="Share Price ($US)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($US Millions)", row=2, col=1)

    fig.update_layout(
        showlegend=False,
        height=900,
        title=stock,
        xaxis_rangeslider_visible=True
    )

    fig.show()

In [ ]:
make_graph(tesla_data, tesla_revenue, 'Tesla')

In [ ]:
make_graph(gme_data, gme_revenue, 'GameStop')